# Section 01: Digital Transformation with Google Cloud

**Companion notebook for**: `01-digital-transformation.html`

## Learning Objectives
- Explore Google Cloud services and regions programmatically
- Understand IaaS / PaaS / SaaS through hands-on examples
- Compare cloud service models and pricing
- Work with the `gcloud` CLI and Python SDK
- Examine the shared responsibility model in practice

## Prerequisites
- A Google Cloud project with billing enabled (optional — mock outputs provided)
- Python 3.10+

---
## Setup & Dependencies

In [ ]:
%pip install -q google-cloud-compute google-cloud-resource-manager google-cloud-billing

In [ ]:
import os
import json

PROJECT_ID = os.environ.get("GCP_PROJECT_ID", "your-project-id")
REGION = "us-central1"
ZONE = "us-central1-a"

print(f"Project ID: {PROJECT_ID}")
print(f"Region:     {REGION}")
print(f"Zone:       {ZONE}")

if PROJECT_ID == "your-project-id":
    print("\nWARNING: Set GCP_PROJECT_ID or replace 'your-project-id' above.")
    print("Mock outputs will be used for API calls.")

---
## Section 1: Exploring Google Cloud Regions and Zones

Google Cloud's infrastructure spans 40+ regions with 120+ zones.
Each region is an independent geographic area, and each zone is
an isolated deployment area within a region.

In [ ]:
# Explore Google Cloud regions and zones
try:
    from google.cloud import compute_v1
    client = compute_v1.RegionsClient()
    regions = list(client.list(project=PROJECT_ID))
    print(f"Found {len(regions)} regions in project {PROJECT_ID}\n")
    print(f"{'Region':<25} {'Status':<10} {'Zones'}")
    print("-" * 70)
    for r in sorted(regions, key=lambda x: x.name)[:15]:
        zones = [z.split('/')[-1] for z in r.zones]
        print(f"{r.name:<25} {r.status:<10} {', '.join(zones)}")
except Exception as e:
    print(f"API call failed: {e}")
    print("\nMock output — Google Cloud Regions (subset):")
    mock_regions = [
        ("asia-east1", "UP", "asia-east1-a, asia-east1-b, asia-east1-c"),
        ("asia-southeast1", "UP", "asia-southeast1-a, asia-southeast1-b, asia-southeast1-c"),
        ("europe-west1", "UP", "europe-west1-b, europe-west1-c, europe-west1-d"),
        ("europe-west4", "UP", "europe-west4-a, europe-west4-b, europe-west4-c"),
        ("us-central1", "UP", "us-central1-a, us-central1-b, us-central1-c, us-central1-f"),
        ("us-east1", "UP", "us-east1-b, us-east1-c, us-east1-d"),
        ("us-west1", "UP", "us-west1-a, us-west1-b, us-west1-c"),
        ("southamerica-east1", "UP", "southamerica-east1-a, southamerica-east1-b, southamerica-east1-c"),
    ]
    print(f"\n{'Region':<25} {'Status':<10} {'Zones'}")
    print("-" * 70)
    for name, status, zones in mock_regions:
        print(f"{name:<25} {status:<10} {zones}")

---
## Section 2: Cloud Service Models — IaaS, PaaS, SaaS

Understanding the three cloud service models is fundamental to the CDL exam.
Let's explore them through Google Cloud examples.

In [ ]:
# Service model comparison — study reference
service_models = {
    "IaaS (Infrastructure as a Service)": {
        "description": "Rent raw compute, storage, networking. You manage OS + apps.",
        "gcp_services": ["Compute Engine", "Cloud Storage", "VPC", "Persistent Disk"],
        "customer_manages": ["Operating System", "Middleware", "Runtime", "Applications", "Data"],
        "google_manages": ["Physical hardware", "Hypervisor", "Network fabric", "Data center"],
        "analogy": "Renting a plot of land — you build and maintain the house",
    },
    "PaaS (Platform as a Service)": {
        "description": "Managed runtime for apps. Deploy code, Google handles the rest.",
        "gcp_services": ["App Engine", "Cloud Run", "Cloud Functions", "Cloud SQL"],
        "customer_manages": ["Application code", "Data"],
        "google_manages": ["OS", "Runtime", "Scaling", "Patching", "Infrastructure"],
        "analogy": "Renting an apartment — you furnish it, landlord maintains the building",
    },
    "SaaS (Software as a Service)": {
        "description": "Fully managed applications consumed through a browser.",
        "gcp_services": ["Google Workspace", "Looker", "Google Meet", "AppSheet"],
        "customer_manages": ["Data", "User access"],
        "google_manages": ["Everything else"],
        "analogy": "Staying at a hotel — everything is provided, you just use it",
    },
}

for model, info in service_models.items():
    print(f"\n{'='*60}")
    print(f"  {model}")
    print(f"{'='*60}")
    print(f"  {info['description']}")
    print(f"  Analogy: {info['analogy']}")
    print(f"\n  GCP Services: {', '.join(info['gcp_services'])}")
    print(f"  You manage:    {', '.join(info['customer_manages'])}")
    print(f"  Google manages: {', '.join(info['google_manages'])}")

---
## Section 3: Compute Engine Basics (IaaS Example)

Compute Engine provides virtual machines. Let's explore machine types
and understand how IaaS works in practice.

In [ ]:
# List available machine types in a zone
try:
    from google.cloud import compute_v1
    client = compute_v1.MachineTypesClient()
    machine_types = list(client.list(project=PROJECT_ID, zone=ZONE))
    
    print(f"Machine types in {ZONE} (showing first 15):\n")
    print(f"{'Name':<25} {'vCPUs':>6} {'Memory (GB)':>12} {'Family'}")
    print("-" * 60)
    for mt in sorted(machine_types, key=lambda x: x.name)[:15]:
        mem_gb = mt.memory_mb / 1024
        print(f"{mt.name:<25} {mt.guest_cpus:>6} {mem_gb:>12.1f}")
except Exception as e:
    print(f"API call failed: {e}")
    print("\nMock output — Common Machine Types:")
    mock_types = [
        ("e2-micro", 2, 1.0, "General (shared)"),
        ("e2-small", 2, 2.0, "General (shared)"),
        ("e2-medium", 2, 4.0, "General (shared)"),
        ("e2-standard-2", 2, 8.0, "General"),
        ("e2-standard-4", 4, 16.0, "General"),
        ("n2-standard-2", 2, 8.0, "General"),
        ("n2-standard-8", 8, 32.0, "General"),
        ("c2-standard-4", 4, 16.0, "Compute-optimized"),
        ("m2-ultramem-208", 208, 5888.0, "Memory-optimized"),
        ("a2-highgpu-1g", 12, 85.0, "Accelerator (GPU)"),
    ]
    print(f"\n{'Name':<25} {'vCPUs':>6} {'Memory (GB)':>12} {'Family'}")
    print("-" * 65)
    for name, cpus, mem, family in mock_types:
        print(f"{name:<25} {cpus:>6} {mem:>12.1f} {family}")

---
## Section 4: Cloud Pricing and TCO Analysis

Total Cost of Ownership (TCO) compares the full cost of on-premises
infrastructure vs. cloud. Let's build a simple TCO calculator.

In [ ]:
# Simple TCO comparison: On-premises vs. Google Cloud

def calculate_on_prem_tco(servers, years=3):
    """Estimate on-premises TCO for a given number of servers over N years."""
    hardware_cost = servers * 8000  # $8K per server
    facility_cost_annual = servers * 2400  # Power, cooling, space per server/year
    admin_cost_annual = max(1, servers // 20) * 120000  # 1 admin per 20 servers, $120K salary
    software_licenses_annual = servers * 1200  # OS, monitoring, etc.
    maintenance_annual = hardware_cost * 0.10  # 10% of hardware cost
    
    total_annual = facility_cost_annual + admin_cost_annual + software_licenses_annual + maintenance_annual
    total_tco = hardware_cost + (total_annual * years)
    
    return {
        "hardware (one-time)": hardware_cost,
        "facility (annual)": facility_cost_annual,
        "admin (annual)": admin_cost_annual,
        "licenses (annual)": software_licenses_annual,
        "maintenance (annual)": maintenance_annual,
        "total_annual": total_annual,
        "total_tco": total_tco,
    }

def calculate_cloud_tco(servers, years=3):
    """Estimate Google Cloud TCO for equivalent workload."""
    # n2-standard-4 equivalent, ~$0.20/hr on-demand
    hourly_rate = 0.20
    # Assume 70% avg utilization (cloud advantage: right-sizing)
    effective_servers = servers * 0.7
    monthly_compute = effective_servers * hourly_rate * 730  # hours/month
    # Apply 30% sustained use discount
    monthly_compute *= 0.70
    
    # Networking and storage overhead (~20%)
    monthly_total = monthly_compute * 1.20
    
    annual_cost = monthly_total * 12
    total_tco = annual_cost * years
    
    return {
        "monthly_compute": monthly_compute,
        "monthly_total": monthly_total,
        "annual_cost": annual_cost,
        "total_tco": total_tco,
    }

# Compare for 50 servers over 3 years
servers = 50
years = 3

on_prem = calculate_on_prem_tco(servers, years)
cloud = calculate_cloud_tco(servers, years)

print(f"TCO Comparison: {servers} servers over {years} years")
print("=" * 50)
print(f"\nOn-Premises:")
print(f"  Hardware (one-time): ${on_prem['hardware (one-time)']:>12,.0f}")
print(f"  Annual operating:    ${on_prem['total_annual']:>12,.0f}")
print(f"  {years}-year TCO:         ${on_prem['total_tco']:>12,.0f}")
print(f"\nGoogle Cloud:")
print(f"  Monthly cost:        ${cloud['monthly_total']:>12,.0f}")
print(f"  Annual cost:         ${cloud['annual_cost']:>12,.0f}")
print(f"  {years}-year TCO:         ${cloud['total_tco']:>12,.0f}")
savings = on_prem['total_tco'] - cloud['total_tco']
pct = (savings / on_prem['total_tco']) * 100
print(f"\nCloud savings: ${savings:,.0f} ({pct:.0f}%)")

---
## Section 5: Shared Responsibility Model

Security is shared between Google and the customer. The split depends
on the service model (IaaS vs PaaS vs SaaS).

In [ ]:
# Shared Responsibility Matrix — study reference

responsibility_matrix = {
    "Layer": [
        "Data & Content",
        "Access & Identity (IAM)",
        "Application Code",
        "Operating System",
        "Network Config",
        "Hardware & Facilities",
    ],
    "IaaS (Compute Engine)": [
        "CUSTOMER",
        "CUSTOMER",
        "CUSTOMER",
        "CUSTOMER",
        "SHARED",
        "GOOGLE",
    ],
    "PaaS (App Engine)": [
        "CUSTOMER",
        "CUSTOMER",
        "CUSTOMER",
        "GOOGLE",
        "GOOGLE",
        "GOOGLE",
    ],
    "SaaS (Workspace)": [
        "CUSTOMER",
        "CUSTOMER",
        "GOOGLE",
        "GOOGLE",
        "GOOGLE",
        "GOOGLE",
    ],
}

print("Shared Responsibility Model")
print("=" * 75)
print(f"{'Layer':<25} {'IaaS':<15} {'PaaS':<15} {'SaaS':<15}")
print("-" * 75)
for i, layer in enumerate(responsibility_matrix["Layer"]):
    iaas = responsibility_matrix["IaaS (Compute Engine)"][i]
    paas = responsibility_matrix["PaaS (App Engine)"][i]
    saas = responsibility_matrix["SaaS (Workspace)"][i]
    print(f"{layer:<25} {iaas:<15} {paas:<15} {saas:<15}")

print("\nKey takeaway: Customer ALWAYS manages data and access, regardless of model.")

---
## Section 6: NIST Cloud Characteristics

The five essential characteristics of cloud computing, with Google Cloud examples.

In [ ]:
# NIST Cloud Characteristics with GCP examples
nist_characteristics = [
    {
        "name": "On-demand Self-service",
        "definition": "Provision resources via console/API without human interaction with provider",
        "gcp_example": "Create a VM with one gcloud command or console click",
        "command": "gcloud compute instances create my-vm --zone=us-central1-a",
    },
    {
        "name": "Broad Network Access",
        "definition": "Access services over the internet from any device",
        "gcp_example": "Access Cloud Console from laptop, phone, or tablet",
        "command": "https://console.cloud.google.com",
    },
    {
        "name": "Resource Pooling",
        "definition": "Provider serves multiple tenants from shared infrastructure",
        "gcp_example": "Multiple customers' VMs run on same physical hardware (multi-tenant)",
        "command": "(Transparent to user — Google manages pool allocation)",
    },
    {
        "name": "Rapid Elasticity",
        "definition": "Scale resources up and down quickly, appearing unlimited",
        "gcp_example": "Managed Instance Group scales from 2 to 100 VMs based on CPU",
        "command": "gcloud compute instance-groups managed set-autoscaling my-mig --max=100",
    },
    {
        "name": "Measured Service",
        "definition": "Usage is metered, monitored, and billed transparently",
        "gcp_example": "Compute Engine bills per second; BigQuery bills per TB scanned",
        "command": "gcloud billing accounts list",
    },
]

for i, char in enumerate(nist_characteristics, 1):
    print(f"\n{i}. {char['name']}")
    print(f"   Definition: {char['definition']}")
    print(f"   GCP Example: {char['gcp_example']}")
    print(f"   Command:     {char['command']}")

---
## Section 7: Open Source and Portability

Google Cloud's commitment to open source helps avoid vendor lock-in.

In [ ]:
# Google Cloud open-source projects and their managed equivalents
open_source_map = [
    ("Kubernetes", "GKE", "Container orchestration", "CNCF"),
    ("TensorFlow", "Vertex AI", "ML framework", "Apache 2.0"),
    ("Apache Beam", "Dataflow", "Batch/stream processing", "Apache"),
    ("Istio", "Anthos Service Mesh", "Service mesh", "Apache 2.0"),
    ("Knative", "Cloud Run", "Serverless containers", "Apache 2.0"),
    ("Prometheus", "Cloud Monitoring", "Metrics collection", "Apache 2.0"),
    ("Grafana", "Cloud Monitoring Dashboards", "Visualization", "AGPL"),
    ("PostgreSQL", "Cloud SQL / AlloyDB", "Relational database", "PostgreSQL License"),
]

print("Google Cloud Open Source Ecosystem")
print("=" * 80)
print(f"{'Open Source Project':<20} {'GCP Service':<28} {'Purpose':<25} {'License'}")
print("-" * 80)
for oss, gcp, purpose, license_ in open_source_map:
    print(f"{oss:<20} {gcp:<28} {purpose:<25} {license_}")

print("\nKey benefit: Portability — skills and code transfer between GCP and other environments.")
print("Exam keyword: 'avoid vendor lock-in' -> containers + Kubernetes + open-source frameworks")

---
## Summary

In this notebook we covered the core concepts of Section 01:

1. **Google Cloud Infrastructure** — 40+ regions, 120+ zones, private global network.
2. **Service Models** — IaaS (Compute Engine), PaaS (App Engine/Cloud Run), SaaS (Workspace).
3. **TCO Analysis** — Cloud often cheaper due to OpEx model, right-sizing, and discounts.
4. **Shared Responsibility** — Customer always manages data and access; Google manages infrastructure.
5. **NIST Characteristics** — On-demand, broad access, pooling, elasticity, measured.
6. **Open Source** — Kubernetes, TensorFlow, Beam reduce lock-in and increase portability.

**Next notebook**: Section 02 covers data transformation — BigQuery, Cloud Storage, Pub/Sub, Dataflow, and Looker.